In [9]:
from __future__ import annotations

import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import ttest_1samp

from DataUtil import (
    build_experiments,
    DEFAULT_EXPERIMENTS
)

from ObsUtil import (
    OBS_REGISTRY,
    get_obs_file,
    list_obs_sources,
    obs_coverage,
)

In [10]:
@dataclass(frozen=True)
class FileCollectionConfig:
    group: str
    freq: str
    run: str
    obs: str
    period: str
    model_template: str
    ens_prefix: str = "en"
    ens_width: int = 2
    ens_start: int = 1

    def parse_period(self) -> Tuple[int, int, int, int]:
        m = re.match(r"^(\d{4})(\d{2})-(\d{4})(\d{2})$", self.period)
        if not m:
            raise ValueError(f"Bad period '{self.period}', expected 'YYYYMM-YYYYMM'")
        y0, m0, y1, m1 = map(int, m.groups())
        if (y1, m1) < (y0, m0):
            raise ValueError(f"Bad period '{self.period}' (end < start)")
        return y0, m0, y1, m1

    def years(self) -> List[int]:
        y0, _, y1, _ = self.parse_period()
        return list(range(y0, y1 + 1))

    def ens_labels(self, nens: int) -> List[str]:
        return [
            f"{self.ens_prefix}{i:0{self.ens_width}d}"
            for i in range(self.ens_start, nens + self.ens_start)
        ]


class S2SFileCollector:
    """
    Collect obs + model files for S2S / diagnostics.

    s2s_var_dict: dict[model_var] -> obs_var
    """

    def __init__(
        self,
        *,
        exp_list: dict,
        exp_dict: dict,
        obs_registry,
        s2s_var_dict: Dict[str, str],
        get_obs_file_func,
    ):
        self.exp_list = exp_list
        self.exp_dict = exp_dict
        self.obs_registry = obs_registry
        self.s2s_var_dict = s2s_var_dict
        self.get_obs_file = get_obs_file_func

    def resolve_obs_file(self, obs: str, freq: str, year: int, var: Optional[str] = None) -> str:
        return self.get_obs_file(self.obs_registry, obs, freq=freq, year=year, var=var)

    def model_ts_dir(self, run_meta, freq: str) -> str:
        # NOTE: land path for land variables
        return os.path.join(run_meta.lnd_path, "ts", freq)

    def model_files_for_ens(
        self,
        cfg: FileCollectionConfig,
        *,
        run_meta,
        var: str,
        ens: str,
    ) -> List[str]:
        ts_dir = self.model_ts_dir(run_meta, cfg.freq)
        years = cfg.years()
        return [
            os.path.join(
                ts_dir,
                cfg.model_template % {"var": var, "ens": ens, "year": y, "period": cfg.period},
            )
            for y in years
        ]

    def collect_one_var(self, cfg: FileCollectionConfig, *, var: str, obs_var: str, verbose: bool = True):
        years = cfg.years()
        models = self.exp_list[cfg.group]["models"]

        # obs files
        obs_files = [self.resolve_obs_file(cfg.obs, cfg.freq, y, obs_var) for y in years]
        missing_obs = [f for f in obs_files if not os.path.exists(f)]
        if missing_obs:
            if verbose:
                print(f"[MISSING OBS] {var} (obs_var={obs_var})")
                for f in missing_obs:
                    print("  ", f)
            return None

        out_var = {}

        for exp in models:
            run_meta = self.exp_dict[exp]["runs"][cfg.run]
            if run_meta is None:
                if verbose:
                    print(f"SKIP {exp}: no run='{cfg.run}'")
                continue

            nens = int(self.exp_dict[exp]["nens"])
            ts_dir = self.model_ts_dir(run_meta, cfg.freq)

            model_by_ens = {}
            missing_any = False

            for ens in cfg.ens_labels(nens):
                files = self.model_files_for_ens(cfg, run_meta=run_meta, var=var, ens=ens)

                missing = [f for f in files if not os.path.exists(f)]
                if missing:
                    missing_any = True
                    if verbose:
                        print(f"[MISSING MOD] {exp} {var} {ens} (ts_dir={ts_dir})")
                        for f in missing[:5]:
                            print("   ", f)
                        if len(missing) > 5:
                            print(f"   ... {len(missing)-5} more")

                model_by_ens[ens] = files

            out_var[exp] = {"obs": obs_files, "model": model_by_ens, "ts_dir": ts_dir, "nens": nens}

            if verbose and missing_any:
                print(f"[WARN] {exp} {var}: some ensemble members missing files")

        return out_var

    def collect(
        self,
        cfg: FileCollectionConfig,
        *,
        vars_to_process: Optional[List[str]] = None,
        verbose: bool = True,
    ) -> Dict[str, dict]:
        out = {}
        for var, obs_var in self.s2s_var_dict.items():
            if vars_to_process is not None and var not in vars_to_process:
                continue
            res = self.collect_one_var(cfg, var=var, obs_var=obs_var, verbose=verbose)
            if res is None:
                continue
            out[var] = res
        return out


@dataclass(frozen=True)
class SoilBiasFromFilesConfig:
    out_dir: str | Path

    # Variable names
    model_var_name: str = "SOILWATER_10CM"  # model variable (already kg/m2)
    obs_var_name: str = "H2OSOI"            # obs variable (m3/m3)
    out_var_name: str = "SOILWATER_10CM"    # for metadata / file naming

    # Unit conversion (obs m3/m3 -> kg/m2 via rho*depth)
    depth_cm: float = 10.0
    water_density: float = 1000.0

    # If model is already an integrated product, no DZ integration needed
    model_is_integrated: bool = True

    # Only used if model_is_integrated=False
    dz_var: str = "DZSOI"
    lev_dim: str = "levgrnd"
    soilayer_file: str | None = None

    # Masking
    landmask_file: str | None = None
    landmask_var: str = "landfrac"
    landfrac_threshold: float = 0.1
    mask_land: bool = True

    # Significance
    alpha: float = 0.05
    significance_threshold: float | None = None

    # NEW: t-test robustness
    sig_min_n: int = 5          # require at least this many finite ensemble samples per grid cell
    sig_eps_std: float = 1e-12  # treat near-zero variance cells as not-testable

    # Coords
    lat_name: str = "lat"
    lon_name: str = "lon"

    # NEW: time selection robustness
    nearest_tolerance: str = "12H"  # tolerance used for method="nearest" time selection


class SoilBiasAssessorFromCollectedFiles:
    """
    Input expected (for one var):
      var_files[exp]["obs"]   -> list[str] (obs files)
      var_files[exp]["model"] -> dict[ens]-> list[str] (model files)
    """

    def __init__(self, cfg: SoilBiasFromFilesConfig):
        self.cfg = cfg
        self.out_dir = Path(cfg.out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)

        self.landmask = None
        if cfg.landmask_file and os.path.exists(cfg.landmask_file):
            ds_mask = xr.open_dataset(cfg.landmask_file)
            self.landmask = ds_mask[cfg.landmask_var]
            self.landmask = self._to_0_360(self.landmask)

        self.fixed_dz = None
        if (not cfg.model_is_integrated) and cfg.soilayer_file and os.path.exists(cfg.soilayer_file):
            self.fixed_dz = xr.open_dataset(cfg.soilayer_file)[cfg.dz_var]

    def compute_one_date_from_varfiles(
        self,
        var_files: dict,
        *,
        target_date: str | pd.Timestamp,
        obs_key: str,
    ) -> xr.Dataset:
        t = pd.to_datetime(target_date)

        dsets = []
        for exp, info in var_files.items():
            obs_files = info["obs"]
            model_by_ens = info["model"]

            obs_field = self._load_obs_snapshot_from_files(obs_files, t)
            ens_stack = self._load_model_ensemble_snapshot_from_files(model_by_ens, t)

            ens_stack, obs_aligned = self._align_ens_to_obs(ens_stack, obs_field)

            model_mean = ens_stack.mean("ensemble")
            model_std = ens_stack.std("ensemble")
            bias = model_mean - obs_aligned

            sig_mask = self._compute_significance_mask(
                ens_stack=ens_stack,
                obs_field=obs_aligned,
                alpha=self.cfg.alpha,
                threshold=self.cfg.significance_threshold,
            )

            pcc, rmse = self._pcc_rmse(model_mean, obs_aligned)

            ds_exp = xr.Dataset(
                data_vars=dict(
                    obs=obs_aligned,
                    model_mean=model_mean,
                    model_std=model_std,
                    bias=bias,
                    sig_mask=sig_mask,
                    pcc=xr.DataArray(pcc, dims=()),
                    rmse=xr.DataArray(rmse, dims=()),
                ),
                coords={
                    self.cfg.lat_name: obs_aligned[self.cfg.lat_name],
                    self.cfg.lon_name: obs_aligned[self.cfg.lon_name],
                },
                attrs=dict(
                    target_date=str(t.date()),
                    exp=exp,
                    obs=obs_key,
                    model_var=self.cfg.model_var_name,
                    obs_var=self.cfg.obs_var_name,
                    out_var=self.cfg.out_var_name,
                    depth_cm=float(self.cfg.depth_cm),
                    units="kg/m2",
                    alpha=float(self.cfg.alpha),
                    sig_min_n=int(self.cfg.sig_min_n),
                ),
            ).expand_dims(exp=[exp])

            dsets.append(ds_exp)

        ds = xr.concat(dsets, dim="exp").expand_dims(time=[np.datetime64(t.to_datetime64())])
        return ds

    def save(self, ds: xr.Dataset, out_nc: str | Path, overwrite: bool = False) -> Path:
        out_nc = Path(out_nc)
        if out_nc.exists() and (not overwrite):
            raise FileExistsError(f"Output exists: {out_nc}")
        ds.to_netcdf(out_nc)
        return out_nc

    # ---------- helpers ----------

    def _open_mfd(self, files: list[str]) -> xr.Dataset:
        missing = [f for f in files if not os.path.exists(f)]
        if missing:
            raise FileNotFoundError(f"Missing files (showing up to 5): {missing[:5]}")
        return xr.open_mfdataset(files, combine="by_coords")

    def _to_0_360(self, da_or_ds):
        lon_name = self.cfg.lon_name
        if lon_name not in da_or_ds.coords:
            return da_or_ds
        lon = da_or_ds[lon_name]
        if lon.min() < 0 or lon.max() > 360:
            new_lon = (lon % 360).sortby(lon)
            da_or_ds = da_or_ds.assign_coords({lon_name: new_lon}).sortby(lon_name)
        return da_or_ds

    def _sel_time_robust(self, ds: xr.Dataset, t: pd.Timestamp) -> xr.Dataset:
        """
        Prefer exact match; fall back to nearest within tolerance.
        Helps when times are 00:00 vs 12:00 or slightly shifted.
        """
        if "time" not in ds.coords and "time" not in ds.dims:
            return ds
        try:
            return ds.sel(time=t)
        except Exception:
            tol = np.timedelta64(pd.to_timedelta(self.cfg.nearest_tolerance))
            return ds.sel(time=t, method="nearest", tolerance=tol)

    def _integrate_top_layers(self, ds: xr.Dataset) -> xr.DataArray:
        # Only used if model_is_integrated=False
        sm = ds[self.cfg.model_var_name]

        if self.cfg.dz_var in ds:
            dz = ds[self.cfg.dz_var]
            if "time" in dz.dims:
                dz = dz.isel(time=0)
        elif self.fixed_dz is not None:
            dz = self.fixed_dz
        else:
            raise ValueError("No DZSOI in dataset and no external soilayer_file provided.")

        lev = self.cfg.lev_dim
        dz_cumsum = dz.cumsum(dim=lev)
        top_mask = dz_cumsum <= (self.cfg.depth_cm / 100.0)

        sm_top = sm.where(top_mask)
        dz_masked = dz.where(top_mask)

        sm_mass = sm_top * dz_masked * self.cfg.water_density
        integrated = sm_mass.sum(dim=lev)
        integrated.attrs["units"] = "kg/m2"

        if self.cfg.mask_land and (self.landmask is not None):
            integrated = self._to_0_360(integrated)
            integrated = integrated.where(self.landmask > self.cfg.landfrac_threshold)

        return integrated

    def _load_model_ensemble_snapshot_from_files(
        self,
        model_by_ens: dict[str, list[str]],
        target_date: pd.Timestamp,
    ) -> xr.DataArray:
        ens_fields = []
        ens_labels = []
        for ens, files in model_by_ens.items():
            ds = self._open_mfd(files)
            ds = self._to_0_360(ds)
            ds_day = self._sel_time_robust(ds, target_date)

            if self.cfg.model_is_integrated:
                mod = ds_day[self.cfg.model_var_name]
            else:
                mod = self._integrate_top_layers(ds_day)

            mod = self._to_0_360(mod).squeeze()

            if self.cfg.mask_land and (self.landmask is not None):
                mod = mod.where(self.landmask > self.cfg.landfrac_threshold)

            ens_fields.append(mod)
            ens_labels.append(ens)

        out = xr.concat(ens_fields, dim="ensemble")
        out = out.assign_coords(ensemble=np.array(ens_labels, dtype=str))
        return out

    def _load_obs_snapshot_from_files(self, obs_files: list[str], target_date: pd.Timestamp) -> xr.DataArray:
        ds = self._open_mfd(obs_files)
        ds_day = self._sel_time_robust(ds, target_date)

        obs = ds_day[self.cfg.obs_var_name]

        # mask missing
        fill = obs.attrs.get("_FillValue", None)
        if fill is None:
            fill = obs.attrs.get("missing_value", -9999)
        obs = obs.where(obs != fill)

        obs = self._to_0_360(obs)

        # Convert volumetric (m3/m3) -> kg/m2 using effective depth
        obs = obs * self.cfg.water_density * (self.cfg.depth_cm / 100.0)
        obs.attrs["units"] = "kg/m2"

        if self.cfg.mask_land and (self.landmask is not None):
            obs = obs.where(self.landmask > self.cfg.landfrac_threshold)

        return obs

    def _align_ens_to_obs(self, ens_stack: xr.DataArray, obs: xr.DataArray):
        if (ens_stack.sizes.get(self.cfg.lat_name) != obs.sizes.get(self.cfg.lat_name)) or \
           (ens_stack.sizes.get(self.cfg.lon_name) != obs.sizes.get(self.cfg.lon_name)):
            ens_stack = ens_stack.interp_like(obs)
        return ens_stack, obs

    def _compute_significance_mask(
        self,
        ens_stack: xr.DataArray,
        obs_field: xr.DataArray,
        alpha: float,
        threshold: float | None,
    ) -> xr.DataArray:
        """
        Robust 1-sample t-test on ensemble differences (ens - obs):
          - require at least cfg.sig_min_n finite ensemble samples per grid cell
          - skip cells with ~zero variance (cfg.sig_eps_std)
          - mark not-testable cells as NaN in sig_mask
        """
        diff = ens_stack - obs_field
        arr = diff.transpose("ensemble", self.cfg.lat_name, self.cfg.lon_name).values

        n_valid = np.sum(np.isfinite(arr), axis=0)
        std = np.nanstd(arr, axis=0)

        pvals = np.ones_like(std, dtype=float)  # default: not significant
        ok = (n_valid >= self.cfg.sig_min_n) & (std > self.cfg.sig_eps_std)

        if np.any(ok):
            # compute once, then apply ok mask
            _, p_all = ttest_1samp(arr, popmean=0.0, axis=0, nan_policy="omit")
            pvals[ok] = p_all[ok]

        sig = xr.DataArray(
            (pvals < alpha),
            coords={
                self.cfg.lat_name: obs_field[self.cfg.lat_name],
                self.cfg.lon_name: obs_field[self.cfg.lon_name],
            },
            dims=(self.cfg.lat_name, self.cfg.lon_name),
        )

        # not-testable cells -> NaN (cleaner than warnings/garbage)
        sig = sig.where(ok)

        if threshold is not None:
            mean_bias = (ens_stack.mean("ensemble") - obs_field)
            sig = sig.where(np.abs(mean_bias) > threshold)

        return sig

    def _pcc_rmse(self, model_mean: xr.DataArray, obs: xr.DataArray) -> tuple[float, float]:
        mv = model_mean.values.ravel()
        ov = obs.values.ravel()
        valid = np.isfinite(mv) & np.isfinite(ov)
        if valid.sum() < 2:
            return np.nan, np.nan
        pcc = float(np.corrcoef(mv[valid], ov[valid])[0, 1])
        rmse = float(np.sqrt(((mv[valid] - ov[valid]) ** 2).mean()))
        return pcc, rmse
        

In [11]:
# ===========================================
# Main (final revised): process ALL groups + ALL experiments in each group
#   - skip existing outputs by default (no crash)
# ===========================================
if __name__ == "__main__":
    data_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_output/som_error"
    land_mask = "/compyfs/zhan391/acme_init/lnd_sea_mask/landmask_1x1.nc"
    soilayer_file = "/compyfs/zhan391/acme_init/lnd_sea_mask/dzsoi_elm.nc"
    os.makedirs(out_path, exist_ok=True)

    exp_list = {
        "Jan2012": dict(
            models=["CTRL10-S0", "CAPT10-S0", "DART20-S0", "DART40-S0"],
            period="201201-201203",
            season="Winter",
            init_month=1,
        ),
        "Jun2012": dict(
            models=["CTRL10-S1", "CAPT10-S1", "DART40-S1"],
            period="201206-201208",
            season="Summer",
            init_month=6,
        ),
    }

    # model_var -> obs_var
    obs_key = "ESA_CCI"
    lnd_var_map = {"SOILWATER_10CM": "H2OSOI"}  # model var -> obs var

    landmask_var = "landfrac"

    # settings
    model_is_integrated = True
    depth_cm = 10.0
    alpha = 0.05
    significance_threshold = None

    # NEW: robust significance config
    sig_min_n = 5
    sig_eps_std = 1e-12

    # NEW: time selection tolerance (nearest)
    nearest_tolerance = "12H"

    freq = "daily"
    run = "fc"

    # collector settings (must match filenames!)
    model_template = "%(var)s.%(ens)s.%(year)d.nc"
    ens_prefix = "EN"
    ens_width = 2
    ens_start = 1

    exp_dict = build_experiments(data_path)

    collector = S2SFileCollector(
        exp_list=exp_list,
        exp_dict=exp_dict,
        obs_registry=OBS_REGISTRY,
        s2s_var_dict=lnd_var_map,
        get_obs_file_func=get_obs_file,
    )

    for group, meta in exp_list.items():
        period = meta["period"]
        init_month = int(meta.get("init_month", 1))
        init_time = f"2012-{init_month:02d}-01"

        cfg_collect = FileCollectionConfig(
            group=group,
            freq=freq,
            run=run,
            obs=obs_key,
            period=period,
            model_template=model_template,
            ens_prefix=ens_prefix,
            ens_width=ens_width,
            ens_start=ens_start,
        )

        all_files = collector.collect(cfg_collect, verbose=True)
        print(f"[{group}] Collected variables: {list(all_files.keys())}")

        for model_var, obs_var in lnd_var_map.items():
            if model_var not in all_files:
                continue

            cfg = SoilBiasFromFilesConfig(
                out_dir=out_path,
                landmask_file=land_mask,
                landmask_var=landmask_var,
                soilayer_file=soilayer_file,          # only used if model_is_integrated=False
                model_var_name=model_var,
                obs_var_name=obs_var,
                out_var_name=model_var,
                model_is_integrated=model_is_integrated,
                depth_cm=depth_cm,
                alpha=alpha,
                significance_threshold=significance_threshold,
                sig_min_n=sig_min_n,
                sig_eps_std=sig_eps_std,
                nearest_tolerance=nearest_tolerance,
            )

            assessor = SoilBiasAssessorFromCollectedFiles(cfg)

            ds = assessor.compute_one_date_from_varfiles(
                all_files[model_var],
                target_date=init_time,
                obs_key=obs_key,
            )

            out_nc = (
                Path(out_path)
                / f"soil_bias_{group}_{freq}_{run}_{obs_key}_{model_var}_{pd.to_datetime(init_time):%Y%m%d}.nc"
            )

            # default: SKIP if exists (no crash)
            if out_nc.exists():
                print(f"[{group}] SKIP (exists): {out_nc}")
                continue

            saved = assessor.save(ds, out_nc, overwrite=False)
            print(f"[{group}] Saved: {saved}")


[Jan2012] Collected variables: ['SOILWATER_10CM']


/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_452206/2083073505.py:429: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  _, p_all = ttest_1samp(arr, popmean=0.0, axis=0, nan_policy="omit")
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:627: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return result_to_tuple(hypotest_fun_out(*samples, **kwds),

[Jan2012] SKIP (exists): /compyfs/zhan391/v3_dart_cda_scratch/diag_output/som_error/soil_bias_Jan2012_daily_fc_ESA_CCI_SOILWATER_10CM_20120101.nc
[Jun2012] Collected variables: ['SOILWATER_10CM']


/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_452206/2083073505.py:429: SmallSampleWarning: After omitting NaNs, one or more axis-slices of one or more sample arguments is too small; corresponding elements of returned arrays will be NaN. See documentation for sample size requirements.
  _, p_all = ttest_1samp(arr, popmean=0.0, axis=0, nan_policy="omit")
/qfs/projects/ops/rh7/apps/E3SM/conda_envs/e3smu_1_12_0/compy/conda/envs/e3sm_unified_1.12.0_login/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:627: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return result_to_tuple(hypotest_fun_out(*samples, **kwds),

[Jun2012] SKIP (exists): /compyfs/zhan391/v3_dart_cda_scratch/diag_output/som_error/soil_bias_Jun2012_daily_fc_ESA_CCI_SOILWATER_10CM_20120601.nc
